In [5]:
import pickle

temp1 = '/workspace/inference/Data/hard_label/All_Test.pickle'
with open(temp1,"rb") as fw:
    load_paths = pickle.load(fw)  

load_paths

{'image': ['normal/DCM/10006490.dcm',
  'normal/DCM/10006327.dcm',
  'normal/DCM/10007089.dcm',
  'normal/DCM/10009950.dcm',
  'normal/DCM/10008364.dcm',
  'normal/DCM/10007907.dcm',
  'normal/DCM/10004000.dcm',
  'normal/DCM/10007173.dcm',
  'normal/DCM/10006775.dcm',
  'normal/DCM/10002252.dcm',
  'normal/DCM/10004130.dcm',
  'normal/DCM/10009546.dcm',
  'normal/DCM/10009125.dcm',
  'normal/DCM/10008091.dcm',
  'normal/DCM/10001225.dcm',
  'normal/DCM/10003939.dcm',
  'normal/DCM/10005217.dcm',
  'normal/DCM/10006344.dcm',
  'normal/DCM/10003726.dcm',
  'normal/DCM/10002826.dcm',
  'normal/DCM/10008978.dcm',
  'normal/DCM/10008803.dcm',
  'normal/DCM/10005340.dcm',
  'normal/DCM/10000651.dcm',
  'normal/DCM/10002304.dcm',
  'normal/DCM/10002131.dcm',
  'normal/DCM/10001021.dcm',
  'normal/DCM/10002757.dcm',
  'normal/DCM/10006683.dcm',
  'normal/DCM/10001686.dcm',
  'normal/DCM/10008849.dcm',
  'normal/DCM/10001441.dcm',
  'normal/DCM/10003650.dcm',
  'normal/DCM/10009414.dcm',
  'no

In [3]:
import os
import pickle

# ✅ 설정
data_root = "/workspace/dataset/test"  # 현재 Docker 컨테이너 내 경로
output_pickle = "./mjkim1_FM.pickle"

# 폴더명 기반 라벨 매핑
label_map = {
    "normal": 0,
    "cardiomegaly": 1,
    "advanced_tuberculosis": 2,
    "nodule": 3,
    "active_tuberculosis": 4,
    "consolidation": 5,
    "interstitial_opacity": 6,
    "pleural_effusion": 7,
    "pneumothorax": 8,
    "atelectasis": 9,
    "mediastinal_widening": 10,
    "support_device": 11,
    "pleural_calcification": 12,
    "pneumoperitoneum": 13
}

image_list = []
label_list = []

for folder in os.listdir(data_root):
    folder_path = os.path.join(data_root, folder)
    if not os.path.isdir(folder_path):
        continue

    label = label_map.get(folder)
    if label is None:
        print(f"⚠️ '{folder}'은 label_map에 없음 → 무시")
        continue

    for file in os.listdir(folder_path):
        if file.lower().endswith((".dcm", ".dicom", ".jpg", ".jpeg", ".png")):
            rel_path = os.path.join(folder, file)
            image_list.append(rel_path)
            label_list.append(label)

# 저장
output_data = {
    "image": image_list,
    "label": label_list
}

with open(output_pickle, "wb") as f:
    pickle.dump(output_data, f)

print(f"✅ Pickle 저장 완료: {output_pickle} (총 {len(image_list)}개 파일)")


✅ Pickle 저장 완료: ./mjkim1_FM.pickle (총 1491개 파일)


In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import torch
from torch.utils.data import DataLoader, Dataset
from torch.nn.functional import one_hot
from torchvision import transforms, utils
from torchvision.models.segmentation import fcn_resnet50
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pickle
import pathlib
import random
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from IPython.display import display, clear_output
from PIL import Image
# from torchsummaryX import summary
from sklearn.utils.class_weight import compute_class_weight
from collections import OrderedDict 
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
# import seaborn as sns
import joblib
from sklearn.metrics import accuracy_score # 정확도 함수
import sklearn.metrics
import math
import pandas as pd

import segmentation_models_pytorch as smp
from multi_preprocessing import *
from Weak_set import *
######################################################
DATA_PATH = '/mnt/promedius_dataset/RSNA2024/mjkim1/normal-triage/data'
######################################################

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.set_num_threads(4)  # 원하는 코어 수로 설정
print(f"Using {torch.get_num_threads()} CPU threads.")

# DEVICE = 'cuda'
DEVICE = torch.device('cpu')
VERSION = 'MTL'
LOAD_PATH = './weight'
LOAD_ROOT_PATH1 = './result/texture/ensemble/imagenet/WeakLabel'
LOAD_ROOT_PATH2 = './result/texture/ensemble/imagenet_gamma(base)/WeakLabel'
LOAD_ROOT_PATH3 = './result/texture/ensemble/new_normal_1121_CLAHE/WeakLabel'

Using 4 CPU threads.


In [5]:
with open("./Data/hard_label/All_Test.pickle","rb") as fw:
    load_paths = pickle.load(fw)  

transform_test = transforms.Compose([
    ToTensor(),
])

externalset =Chest_Single_Data_Generator((1024, 1024), DATA_PATH, load_paths['image'], load_paths['label'], transform=transform_test)

# externalset = CustomDataset((1024, 1024),load_paths['image'], load_paths['label'] ,transform=transform_test)#, mode='BCE')

externalloader = DataLoader(externalset, batch_size=1, shuffle=False, num_workers=8)

print(len(externalset))

100


In [6]:
externalloader

In [7]:
class All_Models(nn.Module):
    def __init__(self, ROOT_PATH):
        super().__init__()
        self.dir_names = ['1_cardiomegaly', '2_advanced_tuberculosis','3_nodule','4_active_tuberculosis',
                          '5_consolidation','6_interstitial_opacity','7_pleural_effusion','8_pneumothorax', 
                          '9_atelectasis', '10_mediastinal_widening', '11_support_device', '12_pleural_calcification', '13_pneumoperitoneum']
        self.MODEL_PATHS = [os.path.join(ROOT_PATH, name) for name in self.dir_names]
        
        self.car_model = torch.load(os.path.join(self.MODEL_PATHS[0], 'weight/best_model_36.pth'))
        self.layer_freeze(self.car_model)
        
        self.adtb_model = torch.load(os.path.join(self.MODEL_PATHS[1], 'weight/last_model.pth'))
        self.layer_freeze(self.adtb_model)
        
        self.nodule_model = torch.load(os.path.join(self.MODEL_PATHS[2], 'weight/last_model.pth'))
        self.layer_freeze(self.nodule_model)
        
        self.actb_model = torch.load(os.path.join(self.MODEL_PATHS[3], 'weight/best_model_124.pth'))
        self.layer_freeze(self.actb_model)
        
        self.consolidation_model = torch.load(os.path.join(self.MODEL_PATHS[4], 'weight/best_model_87.pth'))
        self.layer_freeze(self.consolidation_model)
        
        self.inter_opacity_model = torch.load(os.path.join(self.MODEL_PATHS[5], 'weight/best_model_0.pth'))
        self.layer_freeze(self.inter_opacity_model)
        
        self.pleural_effusion_model = torch.load(os.path.join(self.MODEL_PATHS[6], 'weight/best_model_6.pth'))
        self.layer_freeze(self.pleural_effusion_model)
        
        self.pneumothorax_model = torch.load(os.path.join(self.MODEL_PATHS[7], 'weight/last_model.pth'))
        self.layer_freeze(self.pneumothorax_model)
        
        
        self.atelectasis_model = torch.load(os.path.join(self.MODEL_PATHS[8], 'weight/best_model_4.pth'))
        self.layer_freeze(self.atelectasis_model)
        
        self.mediastinal_widening_model = torch.load(os.path.join(self.MODEL_PATHS[9], 'weight/best_model_20.pth'))
        self.layer_freeze(self.mediastinal_widening_model)
        
        self.support_device_model = torch.load(os.path.join(self.MODEL_PATHS[10], 'weight/best_model_1.pth'))
        self.layer_freeze(self.support_device_model)
        
        
        self.pleural_calcification_model = torch.load(os.path.join(self.MODEL_PATHS[11], 'weight/best_model_38.pth'))
        self.layer_freeze(self.pleural_calcification_model)
        
        self.pneumoperitoneum_model = torch.load(os.path.join(self.MODEL_PATHS[12], 'weight/best_model_43.pth'))
        self.layer_freeze(self.pneumoperitoneum_model)
        
    def layer_freeze(self, model):
        for i, child in enumerate(model.children()):
            if i == 1 or i == 2:
                for param in child.parameters():
                    param.requires_grad = False

    def forward(self, x):
        car_x = self.car_model(x)[1].cpu().detach().numpy()[0]
        adtb_x = self.adtb_model(x)[1].cpu().detach().numpy()[0]
        actb_x = self.actb_model(x)[1].cpu().detach().numpy()[0]
        inter_opacity_x = self.inter_opacity_model(x)[1].cpu().detach().numpy()[0]
        pneumothorax_x = self.pneumothorax_model(x)[1].cpu().detach().numpy()[0]
        nodule_x = self.nodule_model(x)[1].cpu().detach().numpy()[0]
        consolidation_x = self.consolidation_model(x)[1].cpu().detach().numpy()[0]
        pleural_effusion_x = self.pleural_effusion_model(x)[1].cpu().detach().numpy()[0]
        
        atelectasis_x = self.atelectasis_model(x)[1].cpu().detach().numpy()[0]
        mediastinal_widening_x = self.mediastinal_widening_model(x)[1].cpu().detach().numpy()[0]
        support_devicen_x = self.support_device_model(x)[1].cpu().detach().numpy()[0]
        
        pleural_calcificatio_x = self.pleural_calcification_model(x)[1].cpu().detach().numpy()[0]
        pneumoperitoneum_x = self.pneumoperitoneum_model(x)[1].cpu().detach().numpy()[0]
        
        ###### 앙상블 코드 추가
        return [car_x, adtb_x, actb_x, inter_opacity_x, pneumothorax_x, nodule_x, consolidation_x, pleural_effusion_x, atelectasis_x, mediastinal_widening_x, support_devicen_x, pleural_calcificatio_x, pneumoperitoneum_x]

In [ ]:
model = All_Models(LOAD_PATH)
model.eval()
model.to(DEVICE)

In [ ]:
labels = []
pred_labels = []
img_path = []
label = ['cardiomegaly', 'advanced_tuberculosis','nodule','active_tuberculosis',
        'consolidation','interstitial_opacity','pleural_effusion','pneumothorax',
        'atelectasis', 'mediastinal_widening', 'support_device', 'pleural_calcification', 'pneumoperitoneum']

for sample in tqdm(externalloader, bar_format='{l_bar}{bar:10}{r_bar}{bar:-10b}'):
    pred = model(sample[0])
    labels.append(sample[2])
    pred_labels.append(pred)
    img_path.append(sample[3])


In [ ]:
pred_labels2 = np.array(pred_labels)
pred_labels2 = pred_labels2.reshape(pred_labels2.shape[0], -1)
# labels2 = [x.item() for x in labels]
labels3 = []
for i in labels:
    if i[0] == 0:
        labels3.append(0)
    else:
        labels3.append(1)
        
svm = joblib.load('/mnt/promedius_dataset/14class_junsik_240326/svm_weight.pickle')
new_n_pred = svm.predict(pred_labels2)

# for i in range(len(new_n_pred)):
#     print(labels3[i])
# print(round(accuracy_score(labels3,new_n_pred), 2))
# print(confusion_matrix(labels3, new_n_pred))